[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S08/2_Practica/para_casa/S08_05_busqueda_hiperparametros.ipynb)

# S08 · 05 — Buscar buenos hiperparámetros: GridSearchCV y RandomizedSearchCV

🏠 Cuaderno para casa · ⏱ ≈ 40 min · Sprint 8 · M3 (UD 3.2 y 3.6–3.8)

**Contexto TurisData.** Hemos visto que cada modelo tiene «botones» (`max_depth`, `learning_rate`, `C`…). Girarlos a mano es lento y sesgado. Los **buscadores** los prueban de forma sistemática con validación cruzada. Pero ojo: **buscar mucho también puede engañarnos**, y el test debe seguir siendo un examen que se hace **una sola vez**.

**Este cuaderno tiene 4 ejercicios y un mini-reto** y es **solo para casa**: no tiene versión «en clase». Es la guía del paso 3 del reto (la tarde del 11/11): hazlo en casa, a tu ritmo y por orden.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, loguniform, uniform
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings("ignore")
SEMILLA = 42
sns.set_theme(style="whitegrid")

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
cv = StratifiedKFold(3, shuffle=True, random_state=SEMILLA)

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), columnas_num),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
])
print("Listo ✔")

### 👀 A2 · EJEMPLO — Por qué el aleatorio explora mejor que la rejilla

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántos valores distintos del hiperparámetro importante prueba la rejilla y cuántos el aleatorio?
2. ¿Por qué eso favorece al aleatorio si solo un hiperparámetro importa de verdad?
3. ¿Cuándo sigue interesando la rejilla?

In [ ]:
# Por qué el aleatorio explora mejor: 9 pruebas en rejilla vs 9 al azar (dos hiperparámetros)
rng = np.random.default_rng(SEMILLA)
gx, gy = np.meshgrid([0.2, 0.5, 0.8], [0.2, 0.5, 0.8])
fig, ejes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
ejes[0].scatter(gx.ravel(), gy.ravel(), color="#2a6f97")
ejes[0].set(title="Rejilla: 9 pruebas, solo 3 valores de cada uno", xlabel="Hiperparámetro importante", ylabel="Hiperparámetro poco importante")
ejes[1].scatter(rng.random(9), rng.random(9), color="#e63946")
ejes[1].set(title="Aleatorio: 9 pruebas, 9 valores de cada uno", xlabel="Hiperparámetro importante")
plt.tight_layout()
plt.show()

### ✏️ A3 · EJERCICIO 1 de 4 — GridSearchCV con un bosque

**Tu turno:** Define un `Pipeline` `pipe_rf` con `prep` y `RandomForestClassifier(random_state=SEMILLA, n_jobs=2)` (paso llamado `"rf"`) y una rejilla. En un `Pipeline`, los hiperparámetros de un paso se escriben `paso__nombre` (por ejemplo `rf__max_depth`): - `n_estimators`: `[100, 200]`, - `max_depth`: `[4, 8]`, - `min_samples_leaf`: `[10, 40]`. Crea `busqueda_grid = GridSearchCV(pipe_rf, rejilla, cv=cv, scoring="roc_auc")`, ajústala con **train** y guarda en `n_combinaciones` cuántas combinaciones hay (con `len(busqueda_grid.cv_results_["params"])`) y en `mejor_auc_grid` su `best_score_`.

**Debe salir:** `8 combinaciones × 3 folds = 24 entrenamientos en 10 s` y `Mejores parámetros: {'rf__max_depth': 8, 'rf__min_samples_leaf': 40, 'rf__n_estimators': 100} · AUC CV = 0.747` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Cuántas combinaciones y cuántos entrenamientos hay?
2. ¿Qué mejores parámetros salen y qué AUC tienen?
3. ¿Por qué el nombre es `rf__max_depth` y no `max_depth`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: los nombres de la rejilla llevan el prefijo del paso: "rf__max_depth", etc.
# y mide el tiempo con t0 = time.perf_counter() antes de .fit y segundos_grid = time.perf_counter() - t0 después
pipe_rf = ...
rejilla = ...
busqueda_grid = ...
segundos_grid = ...
n_combinaciones = ...
mejor_auc_grid = ...

assert n_combinaciones == 8, "2 × 2 × 2 = 8 combinaciones"
assert 0.65 < mejor_auc_grid < 0.85, "El AUC de validación debería estar entre 0,65 y 0,85"
print(f"{n_combinaciones} combinaciones × 3 folds = {n_combinaciones * 3} entrenamientos en {segundos_grid:.0f} s")
print("Mejores parámetros:", busqueda_grid.best_params_, f"· AUC CV = {mejor_auc_grid:.3f}")
print("✔ Ejercicio 1 correcto")

### ✏️ A4 · EJERCICIO 2 de 4 — RandomizedSearchCV con LightGBM

**Tu turno:** Crea `pipe_lgb` (`prep` + `lgb.LGBMClassifier(n_estimators=200, random_state=SEMILLA, n_jobs=2, verbose=-1)`, paso `"lgb"`) y una búsqueda `busqueda_rand = RandomizedSearchCV(pipe_lgb, distribuciones, n_iter=25, cv=cv, scoring="roc_auc", random_state=SEMILLA)` con estas `distribuciones`: - `lgb__learning_rate`: `loguniform(0.01, 0.3)`, - `lgb__num_leaves`: `randint(3, 40)`, - `lgb__min_child_samples`: `randint(10, 200)`, - `lgb__subsample`: `uniform(0.6, 0.4)` (equivale a valores entre 0,6 y 1,0; recuerda añadir `subsample_freq=1` al modelo). Ajústala con train y guarda `mejor_auc_rand` (`best_score_`).

**Debe salir:** `25 combinaciones aleatorias × 3 folds en 10 s · AUC CV = 0.749` y los mejores parámetros (`learning_rate` 0.019, `num_leaves` 23, `min_child_samples` 199, `subsample` 0.755) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A4).

1. ¿Qué AUC en validación cruzada obtiene la mejor combinación?
2. ¿Qué `learning_rate` y cuántas hojas (`num_leaves`) tiene la mejor?
3. ¿Por qué se usa `loguniform` para el `learning_rate`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A4, **después de intentarlo**.

In [ ]:
# pista: RandomizedSearchCV(pipe_lgb, distribuciones, n_iter=25, ...).fit(X_train, y_train); mide el tiempo como antes
pipe_lgb = ...
distribuciones = ...
busqueda_rand = ...
segundos_rand = ...
mejor_auc_rand = ...

assert len(busqueda_rand.cv_results_["params"]) == 25, "Deben probarse 25 combinaciones"
assert 0.65 < mejor_auc_rand < 0.85
print(f"25 combinaciones aleatorias × 3 folds en {segundos_rand:.0f} s · AUC CV = {mejor_auc_rand:.3f}")
print("Mejores parámetros:", {k: round(v, 3) if isinstance(v, float) else v for k, v in busqueda_rand.best_params_.items()})
print("✔ Ejercicio 2 correcto")

### ✏️ A5 · EJERCICIO 3 de 4 — Configuraciones empatadas: la regla del «1 error estándar»

**Tu turno:** Con `resultados = pd.DataFrame(busqueda_rand.cv_results_)`, guarda en `top5` las 5 mejores filas (columnas `param_lgb__num_leaves`, `param_lgb__learning_rate`, `mean_test_score`, `std_test_score`, ordenadas por `mean_test_score` descendente), en `n_empatadas` cuántas combinaciones tienen una media dentro de **una desviación** (`std_test_score` de la mejor) de la mejor media, y en `mas_simple` el **menor** `num_leaves` entre esas empatadas.

**Debe salir:** `13 de 25 configuraciones están a menos de 1 error estándar de la mejor; la más sencilla usa 4 hojas.` y la tabla `top5` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuántas de las 25 configuraciones están a menos de 1 error estándar de la mejor?
2. ¿Cuál es la más sencilla de las empatadas y cuántas hojas usa?
3. ¿Por qué elegir la sencilla aunque la media sea unas milésimas menor?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
resultados = pd.DataFrame(busqueda_rand.cv_results_)
# pista: sort_values("mean_test_score", ascending=False).head(5); filtra por mean >= mejor_media - std_del_mejor
top5 = ...
n_empatadas = ...
mas_simple = ...

assert len(top5) == 5 and top5["mean_test_score"].is_monotonic_decreasing, "top5: cinco filas ordenadas de mejor a peor"
assert n_empatadas >= 2, "Suele haber varias configuraciones casi empatadas"
assert 3 <= mas_simple <= 39
print(f"{n_empatadas} de 25 configuraciones están a menos de 1 error estándar de la mejor; la más sencilla usa {mas_simple} hojas.")
display(top5.round(3))
print("✔ Ejercicio 3 correcto")

### 👀 A6 · EJEMPLO — AUC según el learning_rate (y las hojas)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿Qué rango de AUC tienen las 25 combinaciones?
2. ¿Qué pasa con las combinaciones de `learning_rate` alto (> 0,15)?
3. ¿Qué conclusión sacas sobre qué hiperparámetro importa más?

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(resultados["param_lgb__learning_rate"].astype(float), resultados["mean_test_score"], c=resultados["param_lgb__num_leaves"].astype(int),
            cmap="viridis", s=50)
plt.colorbar(label="num_leaves")
plt.xscale("log")
plt.title("LightGBM: AUC (validación cruzada) según learning_rate")
plt.xlabel("learning_rate (escala logarítmica)")
plt.ylabel("AUC medio en validación cruzada")
plt.show()

### ✏️ A7 · EJERCICIO 4 de 4 — La nota de la búsqueda frente al test

**Tu turno:** Con el mejor modelo de cada búsqueda (`busqueda_grid.best_estimator_` y `busqueda_rand.best_estimator_`, ya reentrenados con todo train), guarda `auc_test_grid` y `auc_test_rand` (AUC en test con `predict_proba`) y en `optimismo_rand` la diferencia `mejor_auc_rand - auc_test_rand`.

**Debe salir:** `Rejilla + RF: AUC CV 0.747 → test 0.734` y `Aleatorio + LightGBM: AUC CV 0.749 → test 0.740 (diferencia +0.009)` y ✔ Ejercicio 4 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A7).

1. ¿Qué AUC tienen en test la rejilla con el bosque y el aleatorio con LightGBM?
2. ¿Cuánto vale `optimismo_rand` y qué significa?
3. ¿Por qué es poco probable que la diferencia sea mayor aquí?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A7, **después de intentarlo**.

In [ ]:
# pista: best_estimator_.predict_proba(X_test)[:, 1]
auc_test_grid = ...
auc_test_rand = ...
optimismo_rand = ...

assert 0.65 < auc_test_grid < 0.85 and 0.65 < auc_test_rand < 0.85
print(f"Rejilla + RF: AUC CV {mejor_auc_grid:.3f} → test {auc_test_grid:.3f}")
print(f"Aleatorio + LightGBM: AUC CV {mejor_auc_rand:.3f} → test {auc_test_rand:.3f} (diferencia {optimismo_rand:+.3f})")
print("✔ Ejercicio 4 correcto")

### ✏️ A8 · MINI-RETO — Informe de la búsqueda

**Tu turno:** Completa el diccionario `informe` con estas claves (para el **mejor de los dos buscadores según su AUC en validación**): - `"modelo"`: `"random_forest"` o `"lightgbm"`, - `"parametros"`: los `best_params_` correspondientes, - `"auc_validacion"` y `"auc_test"`, - `"n_entrenamientos"`: total de ajustes hechos en su búsqueda (combinaciones × folds), - `"recomendacion"`: una frase (texto) que diga si el resultado te parece fiable y por qué (menciona la diferencia validación–test o los empates).

**Debe salir:** ✔ Mini-reto correcto y el diccionario `informe` (ganador `lightgbm`, 75 entrenamientos, AUC validación 0,749 y test 0,74)

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué buscador gana por AUC de validación y cuántos entrenamientos ha hecho?
2. ¿Qué ganó realmente el aleatorio frente a la rejilla?
3. ¿Te parece fiable el resultado? Razónalo (diferencia validación–test o empates).

In [ ]:
# pista: compara mejor_auc_grid con mejor_auc_rand y rellena el diccionario con los datos del ganador
informe = ...

assert set(informe) == {"modelo", "parametros", "auc_validacion", "auc_test", "n_entrenamientos", "recomendacion"}, "Faltan o sobran claves"
assert informe["modelo"] in {"random_forest", "lightgbm"}
assert informe["n_entrenamientos"] in {24, 75}, "8 × 3 = 24 (rejilla) o 25 × 3 = 75 (aleatorio)"
assert isinstance(informe["recomendacion"], str) and len(informe["recomendacion"]) > 30
display(informe)
print("✔ Mini-reto correcto")

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas y los datos

- **Qué hace:**
    - Carga las herramientas: `pandas`, `numpy`, `matplotlib`, `seaborn`, las distribuciones de `scipy.stats`, los buscadores `GridSearchCV` y `RandomizedSearchCV`, el bosque, `LightGBM` y el preprocesado.
    - `dato(...)` busca un archivo de datos del curso y, si no está, lo descarga del repositorio.
    - Lee `reservas_turisdata.csv` (6000 reservas), crea `temporada_alta` y define `X` (9 numéricas y 6 categóricas) e `y` (si la reserva se cancela). Reparte train y test (80 % / 20 %).
    - `cv` es la validación cruzada de **3** folds (para que las búsquedas no tarden demasiado) y `prep` el preprocesado (imputar nulos y *one-hot*; los árboles no necesitan escalar).
- **Qué debe salir:**
    - `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.

## 1. Parámetros frente a hiperparámetros

#### A2 · Ejemplo — Por qué el aleatorio explora mejor que la rejilla

| | Parámetros | Hiperparámetros |
|---|---|---|
| Quién los fija | El **modelo**, al entrenar | **Nosotros**, antes de entrenar |
| Ejemplos | Coeficientes de la logística, umbrales de un árbol | `max_depth`, `learning_rate`, `C`, `k`, `alpha` |

Un buscador prueba combinaciones de hiperparámetros, evalúa cada una con **validación cruzada** (sobre train) y se queda con la mejor (`refit=True` reentrena con ella todo el train).

- **`GridSearchCV`**: prueba **todas** las combinaciones de una rejilla. Exhaustivo, pero se dispara: 3 valores × 3 valores × 3 valores × 5 folds = 135 entrenamientos.
- **`RandomizedSearchCV`**: prueba `n_iter` combinaciones **al azar** de distribuciones. Explora más valores distintos de los hiperparámetros importantes con el mismo presupuesto.

- **Qué hace:**
    - `np.meshgrid` genera las 9 combinaciones de una rejilla 3 × 3; `rng.random(9)` genera 9 puntos al azar en el cuadrado.
    - Dibuja a la izquierda la rejilla (puntos azules) y a la derecha las 9 pruebas aleatorias (puntos rojos), con un hiperparámetro importante en el eje x y uno poco importante en el eje y.
- **Qué debe salir:**
    - Dos gráficos: a la izquierda, 9 puntos en 3 columnas y 3 filas (solo 3 valores distintos en cada eje); a la derecha, 9 puntos dispersos (9 valores distintos en cada eje).
- **Cómo leerlo:**
    - Si solo importa el hiperparámetro del eje x, la rejilla explora **3 valores distintos** de él (los otros 6 puntos repiten x); el aleatorio explora **9**. Con el mismo coste, mira más valores del que importa.
- **Si falla:**
    - `NameError: name 'np' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos valores distintos del hiperparámetro importante prueba la rejilla y cuántos el aleatorio?* → La rejilla **3** y el aleatorio **9**, con las mismas 9 pruebas.
2. *¿Por qué eso favorece al aleatorio si solo un hiperparámetro importa de verdad?* → Porque cada prueba aleatoria estrena un valor distinto del importante, mientras que la rejilla **repite** los mismos 3 valores en cada fila.
3. *¿Cuándo sigue interesando la rejilla?* → Cuando hay **pocos hiperparámetros y pocos valores** (como en el ejercicio 1: 8 combinaciones): es exhaustiva y barata.

</details>

## 2. GridSearchCV con un bosque aleatorio

#### A3 · Ejercicio 1 — GridSearchCV con un bosque

- **Qué hace:**
    - `pipe_rf` es un `Pipeline` con el preprocesado y el bosque (paso `"rf"`). `rejilla` tiene 2 valores para cada uno de 3 hiperparámetros (2 × 2 × 2 = **8 combinaciones**); los nombres llevan el prefijo `rf__` del paso.
    - `GridSearchCV(pipe_rf, rejilla, cv=cv, scoring="roc_auc").fit(X_train, y_train)` prueba las 8 combinaciones con validación cruzada de 3 folds y reentrena la mejor con todo train.
    - `time.perf_counter()` mide el tiempo (`segundos_grid`). `len(busqueda_grid.cv_results_["params"])` cuenta las combinaciones probadas y `busqueda_grid.best_score_` es el AUC medio de la mejor.
- **Qué debe salir:**
    - `8 combinaciones × 3 folds = 24 entrenamientos en 10 s` (los segundos cambian con el ordenador).
    - `Mejores parámetros: {'rf__max_depth': 8, 'rf__min_samples_leaf': 40, 'rf__n_estimators': 100} · AUC CV = 0.747`.
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `2 × 2 × 2 = 8 combinaciones` → la rejilla no tiene 2 valores en cada uno de los 3 hiperparámetros.
    - `KeyError`/`ValueError: Invalid parameter 'max_depth'` → falta el prefijo del paso: debe ser `rf__max_depth`.
    - `El AUC de validación debería estar entre 0,65 y 0,85` → `mejor_auc_grid` no es `busqueda_grid.best_score_`, o has buscado sobre `X_test`.
    - `NameError: name 'prep' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas combinaciones y cuántos entrenamientos hay?* → **8** combinaciones y 8 × 3 = **24** entrenamientos.
2. *¿Qué mejores parámetros salen y qué AUC tienen?* → `max_depth = 8`, `min_samples_leaf = 40` y `n_estimators = 100`, con AUC **0,747** en validación cruzada.
3. *¿Por qué el nombre es `rf__max_depth` y no `max_depth`?* → Porque `max_depth` es un parámetro del **paso `rf`** dentro del `Pipeline`: la forma es `paso__parametro`.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
pipe_rf = Pipeline([("prep", prep), ("rf", RandomForestClassifier(random_state=SEMILLA, n_jobs=2))])
rejilla = {"rf__n_estimators": [100, 200], "rf__max_depth": [4, 8], "rf__min_samples_leaf": [10, 40]}
t0 = time.perf_counter()
busqueda_grid = GridSearchCV(pipe_rf, rejilla, cv=cv, scoring="roc_auc").fit(X_train, y_train)
segundos_grid = time.perf_counter() - t0
n_combinaciones = len(busqueda_grid.cv_results_["params"])
mejor_auc_grid = busqueda_grid.best_score_
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 3. RandomizedSearchCV con LightGBM

#### A4 · Ejercicio 2 — RandomizedSearchCV con LightGBM

En lugar de listas, damos **distribuciones** (`scipy.stats`): `randint(a, b)` para enteros y `loguniform(a, b)` para valores que varían en órdenes de magnitud (como el `learning_rate`).

- **Qué hace:**
    - `pipe_lgb` es el `Pipeline` con el preprocesado y LightGBM (paso `"lgb"`, con `subsample_freq=1` para que `subsample` funcione).
    - `distribuciones` da para cada hiperparámetro cómo sortear valores: `loguniform(0.01, 0.3)` para el `learning_rate`, `randint(3, 40)` para `num_leaves`, `randint(10, 200)` para `min_child_samples` y `uniform(0.6, 0.4)` para `subsample` (valores entre 0,6 y 1,0).
    - `RandomizedSearchCV(..., n_iter=25, ...)` sortea **25** combinaciones, las evalúa con validación cruzada y se queda con la mejor (`best_score_`).
- **Qué debe salir:**
    - `25 combinaciones aleatorias × 3 folds en 10 s · AUC CV = 0.749` (los segundos cambian con el ordenador).
    - `Mejores parámetros: {'lgb__learning_rate': 0.019, 'lgb__min_child_samples': 199, 'lgb__num_leaves': 23, 'lgb__subsample': 0.755}`.
    - `✔ Ejercicio 2 correcto`.
- **Si falla:**
    - `Deben probarse 25 combinaciones` → falta `n_iter=25` en `RandomizedSearchCV`.
    - `ValueError: Invalid parameter` → los nombres necesitan el prefijo del paso: `lgb__learning_rate`, etc.
    - `AttributeError: ... best_score_` → falta el `.fit(X_train, y_train)` al final de la búsqueda.
    - `NameError: name 'loguniform' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC en validación cruzada obtiene la mejor combinación?* → **0,749**.
2. *¿Qué `learning_rate` y cuántas hojas (`num_leaves`) tiene la mejor?* → `learning_rate ≈ 0,019` y `num_leaves = 23`.
3. *¿Por qué se usa `loguniform` para el `learning_rate`?* → Porque sus valores razonables varían en **órdenes de magnitud** (0,01 a 0,3): con `loguniform` se prueban igual de a menudo 0,01–0,03 que 0,1–0,3. Con una distribución uniforme casi todos serían mayores de 0,1.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
pipe_lgb = Pipeline([("prep", prep), ("lgb", lgb.LGBMClassifier(n_estimators=200, subsample_freq=1,
                                                                random_state=SEMILLA, n_jobs=2, verbose=-1))])
distribuciones = {"lgb__learning_rate": loguniform(0.01, 0.3), "lgb__num_leaves": randint(3, 40),
                  "lgb__min_child_samples": randint(10, 200), "lgb__subsample": uniform(0.6, 0.4)}
t0 = time.perf_counter()
busqueda_rand = RandomizedSearchCV(pipe_lgb, distribuciones, n_iter=25, cv=cv, scoring="roc_auc",
                                   random_state=SEMILLA).fit(X_train, y_train)
segundos_rand = time.perf_counter() - t0
mejor_auc_rand = busqueda_rand.best_score_
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 4. Leer cv_results_: ¿hay un ganador claro?

#### A5 · Ejercicio 3 — Configuraciones empatadas: la regla del «1 error estándar»

Toda la búsqueda queda en `cv_results_`: para cada combinación, la media y la desviación de la puntuación en validación cruzada. Es habitual encontrar **muchas configuraciones casi empatadas**. Una regla práctica es la de **«1 error estándar»**: entre las configuraciones cuya media queda dentro de **una desviación** de la mejor, elegir la **más sencilla** (menos hojas, menos árboles…): rendimiento equivalente, menos riesgo y menos coste.

- **Qué hace:**
    - `resultados = pd.DataFrame(busqueda_rand.cv_results_)` convierte los resultados de las 25 combinaciones en una tabla; `sort_values("mean_test_score", ascending=False)` la ordena de mejor a peor.
    - `top5` toma las 5 primeras filas y las 4 columnas pedidas (`num_leaves`, `learning_rate`, media y desviación).
    - `umbral_1se` es la mejor media menos su desviación; `empatadas` son las combinaciones con media por encima; `n_empatadas` las cuenta.
    - `mas_simple` es el menor `num_leaves` entre las empatadas (`astype(int)` porque `cv_results_` los guarda como objetos).
- **Qué debe salir:**
    - `13 de 25 configuraciones están a menos de 1 error estándar de la mejor; la más sencilla usa 4 hojas.`
    - La tabla `top5`: `num_leaves` 23, 4, 18, 13, 11 con `learning_rate` 0,019 · 0,092 · 0,016 · 0,020 · 0,013, medias 0,749 · 0,749 · 0,749 · 0,748 · 0,747 y desviaciones ≈ 0,011–0,013.
    - `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - Mira la tabla: la 1.ª usa 23 hojas y la 2.ª solo 4, y **rinden igual** (0,749). La diferencia entre las cinco primeras (0,002) es mucho menor que la desviación entre folds (≈ 0,012): están **empatadas**.
- **Si falla:**
    - `top5: cinco filas ordenadas de mejor a peor` → falta `ascending=False` en `sort_values`, o `head(5)`.
    - `Suele haber varias configuraciones casi empatadas` → el filtro de `n_empatadas` no usa «media ≥ mejor media − desviación de la mejor».
    - `assert 3 <= mas_simple <= 39` → `mas_simple` debe ser un entero: el menor `num_leaves` de las empatadas (con `.astype(int)`).
    - `NameError: name 'busqueda_rand' is not defined` → falta el ejercicio 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas de las 25 configuraciones están a menos de 1 error estándar de la mejor?* → **13** de 25.
2. *¿Cuál es la más sencilla de las empatadas y cuántas hojas usa?* → La que usa **4 hojas** (`num_leaves = 4`, `learning_rate ≈ 0,092`), frente a las 23 de la mejor.
3. *¿Por qué elegir la sencilla aunque la media sea unas milésimas menor?* → Porque la diferencia es **ruido** (menor que la desviación entre folds), y un modelo más simple es más barato, más estable y menos propenso a sobreajustar.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
resultados = pd.DataFrame(busqueda_rand.cv_results_)
resultados_ord = resultados.sort_values("mean_test_score", ascending=False)
top5 = resultados_ord[["param_lgb__num_leaves", "param_lgb__learning_rate", "mean_test_score", "std_test_score"]].head(5)
mejor_fila = resultados_ord.iloc[0]
umbral_1se = mejor_fila["mean_test_score"] - mejor_fila["std_test_score"]
empatadas = resultados_ord[resultados_ord["mean_test_score"] >= umbral_1se]
n_empatadas = len(empatadas)
mas_simple = int(empatadas["param_lgb__num_leaves"].astype(int).min())
top5.round(3)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — AUC según el learning_rate (y las hojas)

- **Qué hace:**
    - Dibuja las 25 combinaciones probadas: en el eje x el `learning_rate` (escala logarítmica), en el eje y su AUC medio en validación cruzada, y el color indica `num_leaves`.
- **Qué debe salir:**
    - Un gráfico de 25 puntos: los 17 puntos con `learning_rate` por debajo de 0,1 están entre 0,725 y 0,749; los de `learning_rate` alto (> 0,15) caen hasta 0,68–0,71.
- **Cómo leerlo:**
    - Con `learning_rate` ≥ 0,15 el AUC baja (4 combinaciones, media 0,697) y con `learning_rate` < 0,05 se mantiene (12 combinaciones, media 0,744): el **`learning_rate` bajo es lo importante**; con él, las hojas casi no cambian el resultado.
- **Si falla:**
    - `NameError: name 'resultados' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué rango de AUC tienen las 25 combinaciones?* → De **0,677** (la peor) a **0,749** (la mejor).
2. *¿Qué pasa con las combinaciones de `learning_rate` alto (> 0,15)?* → Son las **peores**: el AUC baja a 0,68–0,71 (media 0,697 frente a 0,744 con `learning_rate` < 0,05).
3. *¿Qué conclusión sacas sobre qué hiperparámetro importa más?* → El **`learning_rate`**: con valores bajos todas rinden parecido. Es la ventaja del aleatorio: explora muchos valores de él.

</details>

## 5. La nota de la búsqueda es optimista

#### A7 · Ejercicio 4 — La nota de la búsqueda frente al test

Cuando se prueban muchas configuraciones, la **mejor puntuación de validación** es en parte **suerte** (la «maldición del ganador»): entre 25 intentos, alguno sale bien por azar. Por eso el resultado honesto es el del **test**, que no se ha usado para elegir nada y **se evalúa una sola vez** con el modelo final. (Aquí lo miramos para aprender cuánto de optimista es la nota de la búsqueda; en el **reto** el test es **otro**, uno nuevo que no habéis visto, y solo se mira con el campeón.)

- **Qué hace:**
    - `busqueda_grid.best_estimator_` y `busqueda_rand.best_estimator_` son los mejores modelos de cada búsqueda, ya reentrenados con todo train.
    - `roc_auc_score(y_test, ....predict_proba(X_test)[:, 1])` mide el AUC de cada uno en test.
    - `optimismo_rand` es la diferencia entre el AUC de validación de la búsqueda aleatoria y el de test.
- **Qué debe salir:**
    - `Rejilla + RF: AUC CV 0.747 → test 0.734`.
    - `Aleatorio + LightGBM: AUC CV 0.749 → test 0.740 (diferencia +0.009)`.
    - `✔ Ejercicio 4 correcto`.
- **Si falla:**
    - `assert 0.65 < auc_test_grid < 0.85 and ...` → no has usado `predict_proba(X_test)[:, 1]` (¿has usado `predict`, que da 0 o 1?) o has usado `X_train`.
    - `NameError: name 'busqueda_grid' is not defined` → faltan los ejercicios 1 y 2.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tienen en test la rejilla con el bosque y el aleatorio con LightGBM?* → **0,734** y **0,740**.
2. *¿Cuánto vale `optimismo_rand` y qué significa?* → **+0,009**: la nota de la búsqueda (0,749) fue algo más alta que la de test (0,740): es el **optimismo** de elegir el mejor de 25.
3. *¿Por qué es poco probable que la diferencia sea mayor aquí?* → Porque las configuraciones están **casi empatadas** (0,744–0,749): elegir la mejor entre ellas apenas «hace trampa». Con configuraciones muy distintas el optimismo sería mayor.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
auc_test_grid = roc_auc_score(y_test, busqueda_grid.best_estimator_.predict_proba(X_test)[:, 1])
auc_test_rand = roc_auc_score(y_test, busqueda_rand.best_estimator_.predict_proba(X_test)[:, 1])
optimismo_rand = mejor_auc_rand - auc_test_rand
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 6. Mini-reto: informe de la búsqueda

#### A8 · Mini-reto — Informe de la búsqueda

- **Qué hace:**
    - `if mejor_auc_rand >= mejor_auc_grid:` elige el ganador: la búsqueda aleatoria con LightGBM (0,749) o la rejilla con el bosque (0,747). Guarda su nombre, su búsqueda y sus dos AUC.
    - `informe` es un diccionario con las seis claves: el modelo, los `best_params_`, los AUC de validación y de test (redondeados), `n_entrenamientos` (combinaciones × 3 folds: `len(busq.cv_results_["params"]) * 3`) y la recomendación (texto tuyo).
    - La recomendación de referencia usa `f"..."` para incluir los dos AUC en la frase.
- **Qué debe salir:**
    - `✔ Mini-reto correcto` y el diccionario: `modelo` `'lightgbm'`, `parametros` los de la mejor combinación, `auc_validacion` 0,749, `auc_test` 0,74, `n_entrenamientos` 75 (25 × 3) y la recomendación.
- **Si falla:**
    - `Faltan o sobran claves` → el diccionario debe tener exactamente `modelo`, `parametros`, `auc_validacion`, `auc_test`, `n_entrenamientos` y `recomendacion`.
    - `8 × 3 = 24 (rejilla) o 25 × 3 = 75 (aleatorio)` → `n_entrenamientos` debe ser combinaciones × 3 folds del buscador ganador.
    - `assert isinstance(informe["recomendacion"], str) and len(...) > 30` → escribe una frase completa (más de 30 caracteres).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué buscador gana por AUC de validación y cuántos entrenamientos ha hecho?* → **El aleatorio con LightGBM** (0,749 frente a 0,747), con **75** entrenamientos (25 × 3). La rejilla hizo 24.
2. *¿Qué ganó realmente el aleatorio frente a la rejilla?* → Casi nada: **0,002** en validación (0,749 frente a 0,747) y 0,006 en test (0,740 frente a 0,734), pero con unas 3 veces más entrenamientos.
3. *¿Te parece fiable el resultado? Razónalo (diferencia validación–test o empates).* → Sí: el test (0,740) es muy parecido a la validación (0,749) y hay **13 configuraciones empatadas**. Mejorar más sería marginal.

</details>

## 7. Para reflexionar

1. La rejilla probó 8 combinaciones y el aleatorio 25. ¿Cuál «costó» más entrenamientos? ¿Cuál exploró más valores distintos de `learning_rate`?
2. Varias configuraciones quedan a menos de un error estándar de la mejor. ¿Por qué elegir la más sencilla?
3. Si tras buscar 500 configuraciones tu AUC de validación es 0,80 y en test 0,74, ¿qué ha pasado?
4. ¿Por qué el test no se usa nunca para elegir hiperparámetros? ¿Qué harías si necesitaras una estimación aún más honesta (pista: validación cruzada anidada)?

<details><summary>Ver respuestas (después de pensarlo)</summary>

1. El aleatorio: 25 × 3 = **75** entrenamientos frente a 8 × 3 = 24 de la rejilla. Y el aleatorio exploró **25 valores distintos** de `learning_rate` (la rejilla del ejercicio 1 ni siquiera lo tenía, porque era un bosque).
2. Porque la diferencia entre ellas es **ruido** (menor que la desviación entre folds). Una configuración más sencilla rinde lo mismo, es más barata, más estable y menos propensa a sobreajustar.
3. Ha ocurrido la **«maldición del ganador»**: de tantas configuraciones, la mejor puntuación de validación es en parte suerte (sobreajuste a la validación). El test (0,74) es la estimación honesta.
4. Porque si el test interviene en la elección, **deja de ser un examen**: el modelo ya habría «visto» sus respuestas y la nota sería optimista. Para una estimación más honesta se usa **validación cruzada anidada**: un bucle externo evalúa y, dentro de cada fold, un bucle interno busca los hiperparámetros.

</details>

## 8. Resumen: qué has aprendido
- Los hiperparámetros se fijan antes de entrenar; los buscadores los prueban con validación cruzada **sobre train**.
- `GridSearchCV` es exhaustivo (caro); `RandomizedSearchCV` explora más con el mismo presupuesto.
- `cv_results_` muestra que suele haber empates: elige lo **simple** dentro de un error estándar.
- La nota de la búsqueda es optimista: el test se evalúa **una sola vez** al final.